# Task 2 — Data Cleaning (Phase 2)
**QFI Internship — Team 2 (Market Risk)**
**Submitted by: Atharv Pratap Singh Chauhan**

**Goal of this task:** turn the raw prices from Task 1 into clean, analysis-ready data.

Steps in this phase:
1. Align trading dates across all stocks (use the benchmark calendar).
2. Handle missing values and remove stale (repeated) prices.
3. Winsorize extreme return errors if needed.
4. Convert prices into **log returns**.
5. Build **equal-weighted portfolio returns**.
6. Compute a **rolling volatility** series.

**Outputs:** clean price matrix, clean return matrix, portfolio return series, rolling volatility.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
print('Libraries loaded.')

## 1. Load the raw data from Task 1
We prefer the files saved by Task 1 (`Raw_Portfolio_Price.csv`), and fall back to the
originally shared files if those are not present yet.

In [ ]:
import os

RAW_FILE   = 'Raw_Portfolio_Price.csv' if os.path.exists('Raw_Portfolio_Price.csv') else '00003070-Raw_Portfolio_Price.csv'
BENCH_FILE = 'Benchmark_prices.csv'    if os.path.exists('Benchmark_prices.csv')    else '00003069-Benchmark_prices.csv'
print('Using price file    :', RAW_FILE)
print('Using benchmark file:', BENCH_FILE)

prices = pd.read_csv(RAW_FILE, index_col=0, parse_dates=True)
prices.index.name = 'Date'

benchmark_prices = pd.read_csv(BENCH_FILE, index_col=0, parse_dates=True)
benchmark_series = benchmark_prices.iloc[:, 0]
benchmark_series.name = 'S&P 500'

tickers = list(prices.columns)
print(f'\nLoaded {prices.shape[0]} rows x {prices.shape[1]} stocks')
prices.head()

## 2. Align trading dates
Every stock should trade on the same set of days as the benchmark. We reindex the price
matrix onto the benchmark's calendar so all series share one common date index.

In [ ]:
# Align to the benchmark trading calendar
prices = prices.reindex(benchmark_series.index)

print('Rows after alignment      :', len(prices))
print('Index matches benchmark   :', prices.index.equals(benchmark_series.index))
print('Duplicate dates in index  :', int(prices.index.duplicated().sum()))

## 3. Handle missing values and stale prices
- **Missing values:** forward-fill (carry the last known price), then drop any leading rows
  that are still empty (before a stock started trading).
- **Stale prices:** a price repeated for several days in a row usually means the feed did not
  update. We flag runs of identical prices so we know the data is genuine.

In [ ]:
missing_before = int(prices.isna().sum().sum())
print('Missing values before cleaning :', missing_before)

# Forward-fill gaps, then drop any rows still fully/partly empty at the very start
prices_clean = prices.ffill().dropna(how='any')

missing_after = int(prices_clean.isna().sum().sum())
print('Missing values after cleaning  :', missing_after)
print('Rows kept                      :', len(prices_clean))

In [ ]:
# Detect stale prices: same price repeated day after day
stale_counts = {}
for t in tickers:
    same_as_prev = prices_clean[t].eq(prices_clean[t].shift(1))
    stale_counts[t] = int(same_as_prev.sum())

stale = pd.Series(stale_counts, name='stale_days').sort_values(ascending=False)
print('Number of days each stock repeated the previous price (stale):')
print(stale.to_string())
print('\n(A handful is normal - e.g. holidays or truly flat days. Large counts would be a red flag.)')

## 4. Convert prices into log returns
We use **log returns**: `r_t = ln(P_t / P_{t-1})`. Log returns add up nicely over time and
are the standard input for volatility and VaR models.

In [ ]:
# Daily log returns for each stock
returns = np.log(prices_clean / prices_clean.shift(1)).dropna()

print('Return matrix :', returns.shape[0], 'days x', returns.shape[1], 'stocks')
returns.head()

### Winsorize extreme return errors (optional safety step)
Genuine market moves can be large, but data errors show up as absurd one-day jumps.
We clip returns at the 0.5% / 99.5% quantiles so a bad tick cannot distort the analysis.
This is light-touch and only affects the very extreme tails.

In [ ]:
lower = returns.quantile(0.005)
upper = returns.quantile(0.995)
returns_wins = returns.clip(lower=lower, upper=upper, axis=1)

n_clipped = int(((returns < lower) | (returns > upper)).sum().sum())
print(f'Return observations clipped by winsorizing: {n_clipped} '
      f'({100*n_clipped/returns.size:.2f}% of all observations)')
returns = returns_wins   # use the winsorized returns downstream

## 5. Build equal-weighted portfolio returns
With no weights specified, we use an **equal-weighted** portfolio: each of the stocks gets
the same weight `1/N`. The portfolio's daily return is the weighted average of stock returns.

In [ ]:
n_assets = len(tickers)
weights  = pd.Series(1.0 / n_assets, index=tickers)   # equal weights
print('Equal weight per stock :', round(weights.iloc[0], 4))
print('Weights sum to         :', round(weights.sum(), 6))

# Portfolio daily return = sum of (weight * stock return)
portfolio_returns = returns.mul(weights, axis=1).sum(axis=1)
portfolio_returns.name = 'Portfolio'
portfolio_returns.head()

In [ ]:
# Compare portfolio vs benchmark (benchmark daily log returns)
benchmark_returns = np.log(benchmark_series / benchmark_series.shift(1)).reindex(portfolio_returns.index).dropna()
common = portfolio_returns.index.intersection(benchmark_returns.index)

plt.figure(figsize=(12, 5))
(1 + portfolio_returns.loc[common]).cumprod().plot(label='Equal-weighted portfolio')
(1 + benchmark_returns.loc[common]).cumprod().plot(label='S&P 500', color='black', alpha=0.7)
plt.title('Growth of 1 unit - Portfolio vs Benchmark')
plt.ylabel('Cumulative growth')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Rolling volatility
Rolling volatility shows how risky the portfolio is over time. We use a 21-day window
(about one trading month) and annualize by multiplying by sqrt(252).

In [ ]:
window = 21   # ~ one trading month
rolling_vol = portfolio_returns.rolling(window).std() * np.sqrt(252)   # annualized

plt.figure(figsize=(12, 4))
rolling_vol.plot(color='crimson')
plt.title(f'Rolling Annualized Volatility (window = {window} days)')
plt.ylabel('Annualized volatility')
plt.tight_layout()
plt.show()

print('Average annualized volatility :', round(rolling_vol.mean(), 4))

## 7. Save the cleaned outputs
These files feed directly into Task 3 (Portfolio Construction) and later the VaR engine.

In [ ]:
prices_clean.to_csv('Clean_Price_Matrix.csv')
returns.to_csv('Clean_Return_Matrix.csv')
portfolio_returns.to_frame().to_csv('Portfolio_Returns.csv')
rolling_vol.to_frame('rolling_vol').to_csv('Rolling_Volatility.csv')
print('Saved: Clean_Price_Matrix.csv, Clean_Return_Matrix.csv, '
      'Portfolio_Returns.csv, Rolling_Volatility.csv')

## Summary
- Aligned all stocks to the benchmark calendar; no missing values remain.
- Checked for stale prices and winsorized extreme return errors.
- Built daily **log returns**, an **equal-weighted portfolio return** series, and a **rolling volatility** series.
- Saved four clean output files for the next phases.

**Next task:** Task 3 — Portfolio Construction (daily portfolio value, P&L, and returns).

*Submitted by Atharv Pratap Singh Chauhan.*